In [ ]:
# Cell 1 — Install + clone repo
!pip install transformers scikit-learn scipy gdown datasets -q

import os, sys, glob
from pathlib import Path

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

sys.path.insert(0, str(REPO / 'delta_system'))
print('Files:', sorted([f.name for f in (REPO/'delta_system').glob('*.py')]))

In [ ]:
# Cell 2 — Train Wikipedia G (skipped if checkpoint already exists)
import math, torch
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizerFast
from datasets import load_dataset

sys.path.insert(0, str(Path('/kaggle/working/multihop-retrieval/delta_system')))
from model  import DeltaSystem
from losses import recon_loss, sparsity_loss, specificity_loss

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR  = Path('/kaggle/working/checkpoints')
CKPT_PATH = CKPT_DIR / 'wiki_model.pt'
CKPT_DIR.mkdir(exist_ok=True)

if CKPT_PATH.exists():
    print(f'Checkpoint already exists: {CKPT_PATH} — skipping training')
else:
    print('No checkpoint found. Training Wikipedia G (8000 pairs, 2000 steps)...')
    print(f'Device: {DEVICE}')

    N_TRAIN=8000; N_EVAL=1000; STEPS=2000; BS=16; LR=1e-4
    LAM_S=1.0; LAM_SPEC=1.0; MARGIN=2.0; MAX_LEN=128

    # Load Wikipedia pairs
    def load_wiki_pairs(n_train=8000, n_eval=1000):
        ds = load_dataset('wikimedia/wikipedia', '20231101.en',
                          split='train', streaming=True)
        pairs, seen = [], 0
        total = n_train + n_eval
        for ex in ds:
            sents = [s.strip() for s in ex['text'].split('\n') if len(s.strip()) > 80]
            for i in range(len(sents)-1):
                A, novel = sents[i], sents[i+1]
                if len(A)>1500 or len(novel)>1500: continue
                pairs.append({'A':A,'B':A+' '+novel,'novel':novel})
                if len(pairs)>=total: break
            if len(pairs)>=total: break
            if len(pairs)%1000==0 and len(pairs)>0:
                print(f'  Loaded {len(pairs)}/{total}...')
        return pairs[:n_train], pairs[n_train:total]

    class PairDS(Dataset):
        def __init__(self,p): self.p=p
        def __len__(self): return len(self.p)
        def __getitem__(self,i): return self.p[i]['A'],self.p[i]['B']

    def make_col(tok):
        def col(batch):
            eA=tok([x[0] for x in batch],max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            eB=tok([x[1] for x in batch],max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            return eA['input_ids'],eA['attention_mask'],eB['input_ids'],eB['attention_mask']
        return col

    train_pairs, eval_pairs = load_wiki_pairs(N_TRAIN, N_EVAL)
    print(f'Train: {len(train_pairs)} | Eval: {len(eval_pairs)}')

    tok   = BertTokenizerFast.from_pretrained('bert-base-uncased')
    model = DeltaSystem().to(DEVICE)
    dl    = DataLoader(PairDS(train_pairs), batch_size=BS, shuffle=True,
                       collate_fn=make_col(tok), num_workers=2, pin_memory=True)
    opt   = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

    model.train(); step=0
    while step < STEPS:
        for batch in dl:
            if step>=STEPS: break
            A_ids,A_mask,B_ids,B_mask=[t.to(DEVICE) for t in batch]
            b=A_ids.size(0)
            logits,delta,delta_0,H_A,_=model(A_ids,A_mask,B_ids,B_mask)
            L_r=recon_loss(logits,B_ids,B_mask)
            L_s=sparsity_loss(delta,B_mask)
            L_spec=torch.tensor(0.,device=DEVICE)
            if b>1:
                idx=[*range(1,b),0]
                lw=model.reconstruct(H_A,A_mask,delta[idx],delta_0[idx],B_ids,B_mask)
                L_spec=specificity_loss(logits,lw,B_ids,B_mask,margin=MARGIN)
            loss=L_r+LAM_S*L_s+LAM_SPEC*L_spec
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
            opt.step(); step+=1
            if step%200==0 or step==1:
                print(f'  step {step:4d}/{STEPS} | ppl={math.exp(min(L_r.item(),20)):.1f} | spec={L_spec.item():.4f}')

    trainable={k:v for k,v in model.state_dict().items() if not k.startswith('bert.')}
    torch.save(trainable, CKPT_PATH)
    print(f'Checkpoint saved: {CKPT_PATH}')

In [ ]:
# Cell 3 — Download ap-matched-sentences.db from your Google Drive folder
import requests, re, shutil, gdown
from pathlib import Path

FOLDER_ID = '1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5'
DB_PATH   = Path('/kaggle/working/newsedits/ap-matched-sentences.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists() and DB_PATH.stat().st_size > 1e6:
    print(f'Already downloaded: {DB_PATH.stat().st_size/1e6:.0f} MB')
else:
    # Try to extract individual file ID from the folder page
    print('Looking up file ID...')
    file_id = None
    try:
        resp = requests.get(
            f'https://drive.google.com/drive/folders/{FOLDER_ID}',
            headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        text = resp.text
        idx  = text.find('ap-matched-sentences')
        if idx > 0:
            window     = text[max(0,idx-500): idx+500]
            candidates = re.findall(r'"(1[A-Za-z0-9_\-]{25,50})"', window)
            if candidates:
                file_id = candidates[0]
                print(f'Found file ID: {file_id}')
    except Exception as e:
        print(f'Page lookup failed: {e}')

    if file_id:
        print('Downloading ap-matched-sentences.db (~406 MB)...')
        gdown.download(id=file_id, output=str(DB_PATH), quiet=False)
    else:
        print('Falling back to full folder download (~4 GB). Deletes .pt files after.')
        TEMP = Path('/kaggle/working/gdrive_temp')
        TEMP.mkdir(exist_ok=True)
        gdown.download_folder(id=FOLDER_ID, output=str(TEMP),
                              quiet=False, remaining_ok=True)
        db_files = list(TEMP.rglob('*.db'))
        if db_files:
            shutil.copy(str(db_files[0]), str(DB_PATH))
            print(f'Copied: {db_files[0].name}')
        else:
            print('ERROR: no .db file found in folder')
        for f in TEMP.rglob('*.pt'):
            f.unlink()

print(f'DB ready: {DB_PATH.exists()} | Size: {DB_PATH.stat().st_size/1e6:.0f} MB')

In [ ]:
# Cell 4 — Run zero-shot NewsEdits evaluation
# Wikipedia G evaluated on AP news revisions — zero training on news data

!python /kaggle/working/multihop-retrieval/delta_system/newsedits_zeroshot_eval.py \
    --db /kaggle/working/newsedits/ap-matched-sentences.db \
    --ckpt /kaggle/working/checkpoints/wiki_model.pt \
    --n 1000 \
    --min_added 2